# Phase 5 Live Extraction Evaluation

This notebook proves the first live extraction-evaluation slice for `onto-canon6`.
It runs the current extractor on a small benchmark slice and keeps three lanes separate:

1. reasonableness/support;
2. structural validation;
3. exact preferred-form canonicalization fidelity.


## Notes

- This notebook makes live `llm_client` calls.
- It uses `case_limit=1` to keep the proof narrow and cheap.
- The result is a proof artifact, not a stable headline benchmark claim.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not locate onto-canon6 repo root from notebook cwd")

for candidate_path in (PROJECT_ROOT / "src", PROJECT_ROOT.parent / "llm_client"):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

from pprint import pprint

from onto_canon6.evaluation import LiveExtractionEvaluationService

service = LiveExtractionEvaluationService()
service.benchmark_fixture


PosixPath('/home/brian/projects/onto-canon6/tests/fixtures/psyop_eval_slice.json')

In [2]:
report = service.run_live_benchmark(case_limit=1)
report.summary.model_dump()


LLM_CLIENT_TIMEOUT_POLICY=ban (first observed in call_llm_structured)


TIMEOUT_DISABLED[call_llm_structured]: timeout=60s ignored (set LLM_CLIENT_TIMEOUT_POLICY=allow to re-enable).


completion_cost failed, using fallback: $0.003264 for 3264 tokens


TIMEOUT_DISABLED[call_llm_structured]: timeout=60s ignored (set LLM_CLIENT_TIMEOUT_POLICY=allow to re-enable).


completion_cost failed, using fallback: $0.002692 for 2692 tokens


{'case_count': 1,
 'reasonableness': {'total_candidates': 2,
  'supported_count': 2,
  'partially_supported_count': 0,
  'unsupported_count': 0,
  'supported_rate': 1.0,
  'acceptable_rate': 1.0},
 'validation': {'valid_count': 0, 'needs_review_count': 1, 'invalid_count': 1},
 'canonicalization': {'expected': 4,
  'observed': 2,
  'matched': 0,
  'precision': 0.0,
  'recall': 0.0,
  'f1': 0.0,
  'missing_signatures': (),
  'unexpected_signatures': ()}}

In [3]:
case = report.cases[0]
candidate_rows = [
    {
        "candidate_index": candidate.candidate_index,
        "predicate": candidate.candidate_import.payload.get("predicate"),
        "support_label": candidate.support_label,
        "validation_status": candidate.validation_status,
        "exact_preferred_match": candidate.exact_preferred_match,
        "claim_text": candidate.candidate_import.claim_text,
        "evidence_spans": [span.model_dump() for span in candidate.candidate_import.evidence_spans],
    }
    for candidate in case.candidate_evaluations
]
candidate_rows


[{'candidate_index': 0,
  'predicate': 'oc:replace_designation',
  'support_label': 'supported',
  'validation_status': 'invalid',
  'exact_preferred_match': False,
  'claim_text': "In 2011, under Admiral Eric Olson’s direction, the term PSYOP was officially replaced with the designation 'Military Information Support Operations' (MISO).",
  'evidence_spans': [{'start_char': 0,
    'end_char': 204,
    'text': "In 2011, under the direction of Admiral Eric Olson, commander of the United States Special Operations Command, the term PSYOP was officially replaced with 'Military Information Support Operations' (MISO)."}]},
 {'candidate_index': 1,
  'predicate': 'oc:hold_command_role',
  'support_label': 'supported',
  'validation_status': 'needs_review',
  'exact_preferred_match': False,
  'claim_text': 'Admiral Eric Olson is commander of the United States Special Operations Command.',
  'evidence_spans': [{'start_char': 32,
    'end_char': 109,
    'text': 'Admiral Eric Olson, commander of t

In [4]:
pprint({
    "case_id": case.case_id,
    "important_missing_facts": case.important_missing_facts,
    "overall_notes": case.overall_notes,
    "extraction_run": case.extraction_run.model_dump(),
    "judge_run": case.judge_run.model_dump(),
    "canonicalization": case.canonicalization.model_dump(),
})


{'canonicalization': {'expected': 4,
                      'f1': 0.0,
                      'matched': 0,
                      'missing_signatures': ('{"predicate":"oc:replace_designation","roles":{"actor":[{"entity_id":"ent:person:eric_olson","entity_type":"oc:person","kind":"entity","name":"Admiral '
                                             'Eric '
                                             'Olson"}],"new_designation":[{"kind":"value","normalized":{"value":"Military '
                                             'Information Support '
                                             'Operations"},"raw":"Military '
                                             'Information Support '
                                             'Operations","value_kind":"string"}],"old_designation":[{"kind":"value","normalized":{"value":"PSYOP"},"raw":"PSYOP","value_kind":"string"}],"subject":[{"entity_id":"ent:operation:psychological_operations","entity_type":"oc:psychological_operation","kind":"ent